In [1]:
# Cell 1 — Imports & Setup
import os, sys, random as py_random, numpy as np
os.chdir('/Users/badisasaisriharsha/Desktop/Glaucoma-Detection-ResCapsNet')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.metrics import roc_auc_score
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
py_random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

Device: mps
PyTorch: 2.12.0


In [14]:
# Cell 2 — Config & Data Loaders
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# --- Config ---
IMG_SIZE    = 256
BATCH_SIZE  = 32
PHASE1_EP   = 15
PHASE2_EP   = 15
LR_HEAD     = 5e-4
LR_FINETUNE = 1e-5
DROPOUT     = 0.05
FOCAL_ALPHA = 0.5649   # from Phase 2b counts
FOCAL_GAMMA = 2.0

TRAIN_DIR = 'data/processed/combined/Train'
VAL_DIR   = 'data/processed/combined/Validation'

# --- ImageNet norm (PyTorch standard) ---
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# --- Albumentations dataset wrapper ---
class AlbumentationsDataset(torch.utils.data.Dataset):
    def __init__(self, image_folder, transform):
        self.dataset   = image_folder
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        img_np = np.array(img)
        aug = self.transform(image=img_np)['image']
        return aug, label

train_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=20, p=0.5),
    A.RandomScale(scale_limit=0.15, p=0.5),
    A.Resize(IMG_SIZE, IMG_SIZE),          # re-fix size after scale (prevents collate mismatch)
    A.RandomBrightnessContrast(brightness_limit=0.3, p=0.5),
    A.ColorJitter(p=0.3),
    A.ElasticTransform(p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# --- Datasets ---
train_raw = ImageFolder(TRAIN_DIR)
val_raw   = ImageFolder(VAL_DIR)

train_ds = AlbumentationsDataset(train_raw, train_aug)
val_ds   = AlbumentationsDataset(val_raw,   val_aug)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"Classes: {train_raw.class_to_idx}")

Train: 1163 | Val: 293
Classes: {'Glaucoma_Negative': 0, 'Glaucoma_Positive': 1}


In [7]:
# Cell 3 — ResCapsNet Architecture
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

def squash(x, dim=-1):
    norm_sq = (x ** 2).sum(dim=dim, keepdim=True)
    norm    = norm_sq.sqrt()
    return (norm_sq / (1.0 + norm_sq)) * (x / (norm + 1e-8))

class StatsNet(nn.Module):
    def forward(self, x):
        mean = x.mean(dim=[2, 3])
        std  = x.std(dim=[2, 3])
        return torch.stack([mean, std], dim=1)  # (B, 2, C)

class CapsuleBranch(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1   = nn.Conv2d(48, 64, kernel_size=3, padding=1)
        self.bn1     = nn.BatchNorm2d(64)
        self.conv2   = nn.Conv2d(64, 16, kernel_size=3, padding=1)
        self.bn2     = nn.BatchNorm2d(16)
        self.stats   = StatsNet()
        self.conv1d1 = nn.Conv1d(2, 8, kernel_size=5, stride=2)
        self.bn1d1   = nn.BatchNorm1d(8)
        self.conv1d2 = nn.Conv1d(8, 1, kernel_size=3)
        self.bn1d2   = nn.BatchNorm1d(1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Conv1d)):
                nn.init.normal_(m.weight, 0, 0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.normal_(m.weight, 1, 0.02)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))      # (B, 16, H, W)
        x = self.stats(x)                         # (B, 2, 16)
        x = F.relu(self.bn1d1(self.conv1d1(x)))  # (B, 8, 6)
        x = self.bn1d2(self.conv1d2(x))           # (B, 1, 4)
        x = x.view(x.size(0), -1)                 # (B, 4)
        return x

class RoutingLayer(nn.Module):
    def __init__(self, in_caps=3, out_caps=1, data_in=4, data_out=4, iters=2):
        super().__init__()
        self.in_caps  = in_caps
        self.out_caps = out_caps
        self.iters    = iters
        self.W = nn.Parameter(torch.randn(out_caps, in_caps, data_out, data_in) * 0.02)

    def forward(self, x):
        # x: (B, in_caps, data_in)
        B = x.size(0)
        # W: (out, in, d_out, d_in) ; x: (B, in, d_in)
        x_exp = x.unsqueeze(1).unsqueeze(-1)   # (B, 1, in, d_in, 1)
        W_exp = self.W.unsqueeze(0)             # (1, out, in, d_out, d_in)
        u_hat = W_exp.matmul(x_exp).squeeze(-1) # (B, out, in, d_out)

        b = torch.zeros(B, self.out_caps, self.in_caps, device=x.device)
        for _ in range(self.iters):
            c = F.softmax(b, dim=1)                      # (B, out, in)
            s = (c.unsqueeze(-1) * u_hat).sum(dim=2)     # (B, out, d_out)
            v = squash(s, dim=-1)
            b = b + (u_hat * v.unsqueeze(2)).sum(dim=-1)
        return v  # (B, out_caps, d_out)

class ResCapsNet(nn.Module):
    def __init__(self, num_classes=1, dropout=DROPOUT):
        super().__init__()
        backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        self.res_extractor = nn.Sequential(*list(backbone.features.children())[:4])

        self.branches = nn.ModuleList([CapsuleBranch() for _ in range(3)])
        self.dropout  = nn.Dropout(dropout)

        # Probe branch output dim
        with torch.no_grad():
            dummy = torch.zeros(1, 48, 32, 32)
            cap_dim = self.branches[0](dummy).shape[-1]
        print(f"Branch output dim: {cap_dim}")

        self.routing    = RoutingLayer(in_caps=3, out_caps=1, data_in=cap_dim, data_out=4, iters=2)
        self.classifier = nn.Linear(4, num_classes)

    def forward(self, x):
        feat = self.res_extractor(x)
        caps = torch.stack([b(feat) for b in self.branches], dim=1)  # (B, 3, cap_dim)
        caps = self.dropout(caps)
        routed = self.routing(caps).squeeze(1)  # (B, 4)
        return self.classifier(routed)          # (B, 1)

# --- Test ---
model = ResCapsNet().to(DEVICE)
dummy = torch.zeros(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
out   = model(dummy)
print(f"Model output shape: {out.shape}")
print(f"Backbone blocks: {len(list(model.res_extractor.children()))}")

Branch output dim: 4
Model output shape: torch.Size([2, 1])
Backbone blocks: 4


In [8]:
# Cell 4 — Focal Loss & Freeze/Unfreeze Utilities

# --- Focal Loss ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        # logits: (B,1), targets: (B,) float
        probs    = torch.sigmoid(logits.squeeze(1))
        targets  = targets.float()
        bce      = F.binary_cross_entropy(probs, targets, reduction='none')
        pt       = torch.where(targets == 1, probs, 1 - probs)
        alpha_t  = torch.where(targets == 1,
                               torch.full_like(pt, self.alpha),
                               torch.full_like(pt, 1 - self.alpha))
        loss     = alpha_t * (1 - pt) ** self.gamma * bce
        return loss.mean()

# --- Freeze entire backbone ---
def freeze_backbone(model):
    for p in model.res_extractor.parameters():
        p.requires_grad = False

# --- Unfreeze last 25% of backbone (features[3] only) ---
def unfreeze_backbone_partial(model):
    # features[3] is the 4th (last) block we kept — ~25% of backbone depth
    last_block = list(model.res_extractor.children())[-1]
    for p in last_block.parameters():
        p.requires_grad = True

# --- Count trainable params ---
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Trainable: {trainable:,} / Total: {total:,}")

# --- Verify ---
freeze_backbone(model)
print("Phase 1 (frozen backbone):")
count_params(model)

unfreeze_backbone_partial(model)
print("Phase 2 (features[3] unfrozen):")
count_params(model)

Phase 1 (frozen backbone):
  Trainable: 111,758 / Total: 275,452
Phase 2 (features[3] unfrozen):
  Trainable: 222,670 / Total: 275,452


In [9]:
# Cell 5 — Training Loop
import time

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_probs, all_labels = 0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        probs = torch.sigmoid(logits.squeeze(1)).detach().cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), auc

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_probs, all_labels = 0, [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, labels)
            total_loss += loss.item()
            probs = torch.sigmoid(logits.squeeze(1)).cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())
    auc = roc_auc_score(all_labels, all_probs)
    return total_loss / len(loader), auc, all_probs, all_labels

def run_phase(model, loader_tr, loader_val, optimizer, criterion,
              scheduler, epochs, phase_name, history):
    best_auc, best_weights, patience_count = 0.0, None, 0
    PATIENCE = 7

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_auc   = train_one_epoch(model, loader_tr, optimizer, criterion)
        val_loss, val_auc, _, _ = evaluate(model, loader_val, criterion)
        scheduler.step(val_auc)
        elapsed = time.time() - t0

        history['tr_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['tr_auc'].append(tr_auc)
        history['val_auc'].append(val_auc)

        print(f"[{phase_name}] Ep {ep:02d}/{epochs} | "
              f"tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} | "
              f"val_loss {val_loss:.4f} val_auc {val_auc:.4f} | "
              f"{elapsed:.0f}s")

        if val_auc > best_auc:
            best_auc     = val_auc
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"  Early stop at epoch {ep} (patience {PATIENCE})")
                break

    # Restore best weights
    model.load_state_dict(best_weights)
    print(f"  [{phase_name}] Best val_AUC: {best_auc:.4f}")
    return best_auc

print("Training loop defined.")

Training loop defined.


In [15]:
# Cell 6 — Run Training (Phase 1 + Phase 2)
os.makedirs('models', exist_ok=True)

history = {'tr_loss': [], 'val_loss': [], 'tr_auc': [], 'val_auc': []}
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)

# --- Phase 1: frozen backbone, train capsule head only ---
print("=" * 55)
print("PHASE 1 — Frozen backbone (15 epochs)")
print("=" * 55)

model = ResCapsNet().to(DEVICE)
freeze_backbone(model)
count_params(model)

optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()),
                 lr=LR_HEAD, betas=(0.9, 0.999))
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.3,
                               patience=3, min_lr=1e-7)

phase1_best = run_phase(model, train_loader, val_loader,
                        optimizer, criterion, scheduler,
                        PHASE1_EP, 'P1', history)

# --- Phase 2: unfreeze features[3], fine-tune ---
print("\n" + "=" * 55)
print("PHASE 2 — Unfreeze features[3] (15 epochs)")
print("=" * 55)

unfreeze_backbone_partial(model)
count_params(model)

optimizer = Adam(filter(lambda p: p.requires_grad, model.parameters()),
                 lr=LR_FINETUNE, betas=(0.9, 0.999))
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.3,
                               patience=3, min_lr=1e-7)

phase2_best = run_phase(model, train_loader, val_loader,
                        optimizer, criterion, scheduler,
                        PHASE2_EP, 'P2', history)

torch.save(model.state_dict(), 'models/rescapsnet_stage_a.pth')
print(f"\nModel saved. Phase1 best AUC: {phase1_best:.4f} | Phase2 best AUC: {phase2_best:.4f}")

PHASE 1 — Frozen backbone (15 epochs)
Branch output dim: 4
  Trainable: 111,758 / Total: 275,452
[P1] Ep 01/15 | tr_loss 0.0935 tr_auc 0.7643 | val_loss 0.1066 val_auc 0.8503 | 15s
[P1] Ep 02/15 | tr_loss 0.0770 tr_auc 0.8542 | val_loss 0.0866 val_auc 0.8572 | 11s
[P1] Ep 03/15 | tr_loss 0.0690 tr_auc 0.8781 | val_loss 0.0722 val_auc 0.8776 | 11s
[P1] Ep 04/15 | tr_loss 0.0647 tr_auc 0.8756 | val_loss 0.0696 val_auc 0.8791 | 11s
[P1] Ep 05/15 | tr_loss 0.0622 tr_auc 0.8858 | val_loss 0.0685 val_auc 0.8960 | 11s
[P1] Ep 06/15 | tr_loss 0.0598 tr_auc 0.8945 | val_loss 0.0673 val_auc 0.8977 | 11s
[P1] Ep 07/15 | tr_loss 0.0585 tr_auc 0.8897 | val_loss 0.0661 val_auc 0.8960 | 11s
[P1] Ep 08/15 | tr_loss 0.0574 tr_auc 0.8907 | val_loss 0.0657 val_auc 0.9037 | 11s
[P1] Ep 09/15 | tr_loss 0.0574 tr_auc 0.8911 | val_loss 0.0705 val_auc 0.9000 | 10s
[P1] Ep 10/15 | tr_loss 0.0567 tr_auc 0.8891 | val_loss 0.0605 val_auc 0.8939 | 10s
[P1] Ep 11/15 | tr_loss 0.0546 tr_auc 0.8979 | val_loss 0.0631 

In [16]:
# Cell 7 — Generate Predictions CSV + Shared Scorer
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              roc_curve, f1_score, precision_score,
                              recall_score, accuracy_score, confusion_matrix)

os.makedirs('results/predictions', exist_ok=True)

# --- 1. Generate val predictions (model already has best weights restored) ---
model.eval()
_, _, val_probs, val_labels = evaluate(model, val_loader, criterion)
val_probs  = np.array(val_probs)
val_labels = np.array(val_labels).astype(int)

# image_id: ImageFolder preserves sample order in val_raw.samples (shuffle=False)
image_ids = [os.path.basename(p) for p, _ in val_raw.samples]
assert len(image_ids) == len(val_labels), "image_id / label length mismatch"

pred_df = pd.DataFrame({
    'image_id':   image_ids,
    'true_label': val_labels,
    'pred_prob':  val_probs,
})
pred_df.to_csv('results/predictions/rescapsnet.csv', index=False)
print(f"Saved → results/predictions/rescapsnet.csv ({len(pred_df)} rows)")

# --- 2. Shared scorer (Youden's J threshold; same metrics as the 5 TF models) ---
def score_model(csv_path, model_name):
    df = pd.read_csv(csv_path)
    y_true = df['true_label'].values.astype(int)
    y_prob = df['pred_prob'].values

    auc    = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)

    # Youden's J optimal threshold from ROC
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    youden = tpr - fpr
    best_thr = thr[np.argmax(youden)]

    y_pred = (y_prob >= best_thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        'model':       model_name,
        'AUC':         round(auc, 4),
        'PR_AUC':      round(pr_auc, 4),
        'Sensitivity': round(sensitivity, 4),
        'Specificity': round(specificity, 4),
        'F1_pos':      round(f1_score(y_true, y_pred), 4),
        'Precision':   round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Accuracy':    round(accuracy_score(y_true, y_pred), 4),
        'Threshold':   round(float(best_thr), 4),
    }

res = score_model('results/predictions/rescapsnet.csv', 'rescapsnet')
print("\nResCapsNet — Stage A (shared scorer):")
for k, v in res.items():
    print(f"  {k:12s}: {v}")

Saved → results/predictions/rescapsnet.csv (293 rows)

ResCapsNet — Stage A (shared scorer):
  model       : rescapsnet
  AUC         : 0.9098
  PR_AUC      : 0.9124
  Sensitivity : 0.7031
  Specificity : 0.9758
  F1_pos      : 0.8108
  Precision   : 0.9574
  Accuracy    : 0.8567
  Threshold   : 0.6217
